<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Fed_Stance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers for Text: The Monetary Policy Stance of Fed Speeches

*Deep Learning in Banking* (Bravo, Maldonado and Óskarsdóttir, Wiley, 2026), Chapter 4: updated case study (2026).

> **New case (2026).** This case replaces the case study printed in Chapter 4 of the book. The original case, which predicts next month's Chicago Fed National Activity Index from the same speeches, is still available and corrected: [open it in Colab](https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Textual_Data.ipynb) or [view it on GitHub](https://github.com/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Textual_Data.ipynb).

When a Federal Reserve official speaks, banks, investors and journalists read the speech for one thing first: does it point to higher interest rates (a *hawkish* stance), to lower rates (a *dovish* stance), or to no change (*neutral*)? A bank's treasury desk, for example, reprices its funding plans on that reading. In this lab we train a transformer to make the same reading, passage by passage, on Fed speeches from 1997 to 2024.

We follow the case study of Chapter 4 and its four algorithms, which load the data (Algorithm 4.1), preprocess the text (Algorithm 4.2), fine-tune DistilBERT (Algorithm 4.3), and apply the model to new speeches (Algorithm 4.4). This edition also brings the case up to 2026. The target comes from the text itself, labelled by a language model and checked against human labels. The models are tested on later years than the ones they learn from. Simple baselines are reported next to every model. Finally, a section trains ModernBERT, an encoder released in December 2024, to show how far the field has moved since DistilBERT (but before jumping to LLMs, as we do in Chapter 6). The last section of the notebook lists what changed from the original case.

**Before you start:** select a GPU runtime (*Runtime → Change runtime type → T4 GPU*). The notebook runs on a free Colab T4. The ModernBERT section is the slowest part, and you can switch it off in its first cell if you prefer.

**Data and licences.** About 80% of the passages come from speeches by members of the Board of Governors, which the Board [places in the public domain](https://www.federalreserve.gov/disclaimer.htm), asking to be cited as the source. The rest come from speeches by Reserve Bank presidents, whose banks allow their material to be reproduced with credit (the [New York Fed](https://www.newyorkfed.org/privacy/termsofuse), the source of about 14% of the passages, also for business purposes). We took the speeches from the [collection of central bankers' speeches](https://www.bis.org/cbspeeches/) of the Bank for International Settlements (BIS), and we credit the Board, the Reserve Banks and the BIS as sources. The stance labels were made for this lab (Section 2) and are shared with the passages. The annotated sentences of [Shah et al. (2023)](https://aclanthology.org/2023.acl-long.368/), licensed CC BY-NC 4.0, were used only to check our labels and are not part of this lab. The federal funds rate of Section 10 is read from [FRED](https://fred.stlouisfed.org/series/FEDFUNDS), the database of the Federal Reserve Bank of St. Louis. The code of this notebook is under the Apache 2.0 licence of the repository.

Use only public data in Colab. Do not upload bank or customer data.

## 0. Setup

We install the version of the Hugging Face libraries this notebook was tested with, and set a fixed random seed before any data are split or any model is created, so that the results can be repeated.

In [ ]:
# Install the tested version of the Hugging Face libraries.
!pip install -q transformers==5.18.0 accelerate

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from torch import nn
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, set_seed)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| device:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')

## 1. Loading the data (Algorithm 4.1)

Algorithm 4.1 loads the speeches and an economic indicator, and builds the label by matching each speech to the indicator of the following month. In this edition the label comes from the speech itself, so the matching is already done, and each row of the file below holds one passage of a Fed speech and its stance label. The passages come from the collection of central bankers' speeches compiled by the Bank for International Settlements, the same source as the speeches file of the original case.

The file has 1,401 passages, one per speech, from January 1997 to December 2024. Its columns are:

- `id`: an identifier for the passage.
- `date`: the date of the speech.
- `label`: the stance, one of `hawkish`, `dovish` or `neutral`. This is the target of the lab.
- `label_4`: the stance with a fourth value, `none`, for passages that turned out to have no monetary policy content (211 of them). The main label counts them as `neutral`, as Shah et al. (2023) do for their sentences. An exercise uses this column.
- `source`: `agreed` if the two labelling models gave the same label, `adjudicated` if a third model settled a disagreement (see Section 2).
- `passage`: the text.

In [ ]:
# Download the passages file (bare file ids work with every gdown version).
!gdown -q 1_SKThVZi95-kzZ7X3eOj0DMYjLqkKOGg    # fed_stance_passages.csv

In [ ]:
df = pd.read_csv('fed_stance_passages.csv', parse_dates=['date'])
df['year'] = df['date'].dt.year
df = df.sort_values(['date', 'id']).reset_index(drop=True)

CLASSES = ['dovish', 'hawkish', 'neutral']
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}
df['y'] = df['label'].map(label2id)

print(df.shape)
print(df['label'].value_counts())
df.head()

Before you look at any model, read one passage of each class. The stance is often in a few words, such as "remove accommodation" or "further easing", while most of the passage describes the economy.

In [ ]:
for c in CLASSES:
    row = df[df['label'] == c].sample(1, random_state=SEED).iloc[0]
    print(f"--- {c.upper()} ({row['date'].date()}) ---")
    print(row['passage'][:900], '...' if len(row['passage']) > 900 else '')
    print()

The stance changes with the policy cycle, as the count of labels per year below shows. Hawkish passages dominate before the rate increases of 2004 to 2006 and 2022 to 2023, and dovish ones during the crisis years and the pandemic. Keep this in mind for Section 4, because it means that a model trained on one period meets a different mix of labels in the next.

In [ ]:
counts = df.groupby(['year', 'label']).size().unstack(fill_value=0)[CLASSES]
counts.plot(kind='bar', stacked=True, figsize=(12, 4), color=['tab:blue', 'tab:red', 'tab:gray'])
plt.title('Stance labels per year')
plt.xlabel('')
plt.ylabel('Passages')
plt.tight_layout()
plt.show()

## 2. Where the labels come from (beyond the book)

Supervised learning is only as good as its labels, so before we train anything we explain how these were made, and how much we trust them.

**Passages.** A speech has thousands of words, while DistilBERT reads 512 tokens (as we saw in Chapter 4). For each speech, we kept the sentences about monetary policy or the economic outlook (sentences with terms such as *inflation*, *federal funds*, *unemployment* or *accommodation*), and took the run of such sentences with the most policy terms, up to 400 words. Speeches with fewer than 80 words of policy text were left out, and so were speeches by officials of other central banks given at Fed events, such as the Jackson Hole symposium.

**Labels.** We wrote an annotation guide adapted from [Shah et al. (2023)](https://aclanthology.org/2023.acl-long.368/), who labelled sentences from FOMC documents. The guide defines the label as the direction of US monetary policy that the passage points to in the coming months, as the speaker presents it on the date of the speech. Explicit statements about policy (rate decisions, guidance, asset purchases) decide first, and economic conditions decide only when the passage has no such statement. Two language models (Claude Sonnet and Claude Opus) labelled every passage independently with this guide, and a third (Claude Fable) decided the 136 passages where they disagreed, without knowing which model had given which label.

**How good are the labels?** We checked the labels in three ways, with the following results.

- The two labelling models agree on 90.3% of the passages (Cohen's kappa 0.86). Almost all disagreements were "neutral" against a direction, and the two models gave opposite directions (hawkish against dovish) only once.
- One of us labelled 20 passages blind. The agreement with the final labels was 60% with four labels and 75% with three, a small sample with a wide interval. Most differences came from rules of the guide (for example, whether a passage that only looks back at past policy has a stance), which we then settled manually.
- On 496 sentences that Shah et al. (2023) labelled by hand, our labelling procedure agrees with the human label 68% of the time (kappa 0.47). This is above a zero-shot GPT-3.5 in their benchmark (weighted F1 0.59 against our 0.67) and below their fine-tuned RoBERTa-large (0.71) and the agreement between their two annotators (91%). Our labeller calls "neutral" more often than their annotators did, and it almost never confuses hawkish with dovish.

In short, the labels are good enough to train on and to compare models, and they are conservative, saying neutral when in doubt. Treat every score below as agreement with these labels, not with an absolute truth. Of course, this model should never be used to make policy decisions, and it is not a substitute for human reading of the speeches. It is, however, a good example of how to train a transformer on textual data in banking and a great exercise for you to play with.

## 3. Preprocessing the text (Algorithm 4.2)

Algorithm 4.2 lowercases the text, splits it into words, and removes punctuation and stopwords. As the chapter discusses, the best preprocessing depends on the task, and [Siino et al. (2024)](https://doi.org/10.1016/j.is.2023.102342) found that combining several steps often hurts transformer models. Stance gives a clear reason for care, because the standard English stopword list removes *not*, *no*, *nor* and *further*, and "no further tightening" means the opposite of "further tightening".

The function below implements Algorithm 4.2 but keeps those words, and we use it to see what preprocessing does to a passage. The models in this lab read the raw passage by default (DistilBERT's tokenizer lowercases the text itself), and an exercise asks you to switch the cleaned text on and compare.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

KEEP = {'not', 'no', 'nor', 'further', 'against', 'above', 'below', 'up', 'down', 'over', 'under', 'more', 'most', 'less', 'few'}
STOP = set(stopwords.words('english')) - KEEP

def clean_text(text):
    """Algorithm 4.2: lowercase, tokenize, drop punctuation and stopwords (keeping words that signal stance)."""
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalnum() and w not in STOP]
    return ' '.join(words)

df['text_clean'] = df['passage'].map(clean_text)
print('RAW:  ', df.loc[0, 'passage'][:300])
print('CLEAN:', df.loc[0, 'text_clean'][:300])

In [ ]:
# How much does Algorithm 4.2 remove?
raw_words = df['passage'].str.split().str.len()
clean_words = df['text_clean'].str.split().str.len()
raw_vocab = {w for t in df['passage'] for w in word_tokenize(t.lower()) if w.isalnum()}
clean_vocab = {w for t in df['text_clean'] for w in t.split()}
print(f'Words per passage: raw {raw_words.mean():.0f}, cleaned {clean_words.mean():.0f} '
      f'({1 - clean_words.sum() / raw_words.sum():.0%} of the words removed)')
print(f'Distinct words: raw {len(raw_vocab):,}, cleaned {len(clean_vocab):,}')
kept = [w for w in sorted(KEEP) if w in clean_vocab]
print('Stance words kept by our version of Algorithm 4.2:', ', '.join(kept))

# Which text do the models read? False: the raw passage (default). True: the cleaned text.
USE_CLEAN_TEXT = False
TEXT = 'text_clean' if USE_CLEAN_TEXT else 'passage'
print('The models read:', TEXT)

## 4. Splitting by time

Algorithm 4.3 splits the data into a training and a test set at random. For speeches, a random split mixes passages from the same weeks into both sets, and the model can learn *when* a passage was written rather than *what* it says. The original case of this chapter showed the danger, with a model that looked useful under a random split and did no better than chance on later years (see the usability test at the end of that notebook). [Xue et al. (2026)](https://arxiv.org/abs/2608.17223) found the same pattern in financial news across 16 text models: random splits inflated the scores 1.1 to 6.5 times compared with splits by time.

A bank would use the model on speeches that have not been given yet, so we test it the same way. The model learns from 1997 to 2016, we choose its settings (such as the best epoch) on 2017 to 2019, and we report its performance once, on 2020 to 2024. The test years include the pandemic, the rate increases of 2022 and 2023, and the first cuts of 2024, a hard test for a model trained on earlier cycles.

In [ ]:
train_df = df[df['year'] <= 2016]
val_df = df[(df['year'] > 2016) & (df['year'] <= 2019)]
test_df = df[df['year'] > 2019]

split_counts = pd.DataFrame({name: part['label'].value_counts() for name, part in
                             [('train 1997-2016', train_df), ('validation 2017-2019', val_df), ('test 2020-2024', test_df)]}).loc[CLASSES]
split_counts.loc['total'] = split_counts.sum()
split_counts

## 5. Baselines (beyond the book)

A model is useful only if it beats something simpler, so we compare every model with three baselines, using the same metrics.

- **Majority class:** always predict the most frequent label of the training years (neutral).
- **Persistence:** predict the label of the previous passage in time. Stance moves slowly, so yesterday's speech says a lot about today's. A bank would know the previous label when the next speech comes, so this baseline is fair.
- **TF-IDF and logistic regression:** the classic text model. Each passage becomes a vector of word and two-word frequencies, weighted by how rare each term is, and a logistic regression learns from it. We choose its regularization on the validation years.

The main metric is the **macro F1**, the average of the F1 scores of the three classes. Accuracy rewards a model that predicts neutral often, because neutral is the largest class, whereas macro F1 gives hawkish and dovish the same weight as neutral.

In [ ]:
results = {}

def evaluate_model(name, y_true, y_pred, plot=True):
    """Store and print accuracy, macro F1 and weighted F1; optionally plot the confusion matrix."""
    r = {'accuracy': accuracy_score(y_true, y_pred),
         'macro F1': f1_score(y_true, y_pred, average='macro', labels=[0, 1, 2], zero_division=0),
         'weighted F1': f1_score(y_true, y_pred, average='weighted', labels=[0, 1, 2], zero_division=0)}
    results[name] = r
    print(f"{name}: accuracy {r['accuracy']:.3f} | macro F1 {r['macro F1']:.3f} | weighted F1 {r['weighted F1']:.3f}")
    if plot:
        ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]), display_labels=CLASSES).plot(cmap='Blues', colorbar=False)
        plt.title(f'{name}, test 2020-2024')
        plt.show()

In [ ]:
# Majority class of the training years.
majority = train_df['y'].mode()[0]
evaluate_model('Majority', test_df['y'], np.full(len(test_df), majority), plot=False)

# Persistence: the label of the previous passage, in date order across all years.
previous = df['y'].shift(1).fillna(majority).astype(int)
evaluate_model('Persistence', test_df['y'], previous[test_df.index])

Persistence is not a bad model at all.

In [ ]:
# TF-IDF and logistic regression; C chosen on the validation years.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df[TEXT])
X_val, X_test = vectorizer.transform(val_df[TEXT]), vectorizer.transform(test_df[TEXT])

best = None
for C in [0.1, 0.3, 1, 3, 10, 30]:
    lr_model = LogisticRegression(C=C, max_iter=5000, class_weight='balanced').fit(X_train, train_df['y'])
    score = f1_score(val_df['y'], lr_model.predict(X_val), average='macro')
    print(f'C = {C:>4}: validation macro F1 {score:.3f}')
    if best is None or score > best[0]:
        best = (score, C, lr_model)
print('Chosen C:', best[1])
tfidf_model = best[2]
evaluate_model('TF-IDF + logistic', test_df['y'], tfidf_model.predict(X_test))

... But TF-IDF is even better. The terms with the largest weights show what the TF-IDF model learned. Many are the vocabulary of policy and of the economy, such as *slack*, *recession* or *accommodation*, which is reassuring. Others, such as *year*, *we* or *will*, say more about how speeches were written in certain years than about their stance, a reminder that a bag of words can learn the era as well as the stance.

In [ ]:
terms = np.array(vectorizer.get_feature_names_out())
for i, c in enumerate(CLASSES):
    top = terms[np.argsort(tfidf_model.coef_[i])[::-1][:12]]
    print(f'{c:8s}:', ', '.join(top))

## 6. Fine-tuning DistilBERT (Algorithm 4.3)

We now follow Algorithm 4.3 step by step, with the Hugging Face libraries. DistilBERT (Sanh et al., 2019) is a smaller version of BERT, 40% smaller and 60% faster while keeping most of its language understanding, and it fits comfortably on a free T4. We use the uncased version, which lowercases the text itself.

**Steps 1 to 5: tokenizer and datasets.** With the seed set in Section 0, we load the tokenizer of the checkpoint, define a function that tokenizes a passage and truncates it at 512 tokens, convert the three splits of Section 4 into Hugging Face datasets, and tokenize them.

In [ ]:
CHECKPOINT = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

def to_dataset(part):
    return Dataset.from_pandas(part[[TEXT, 'y']].rename(columns={TEXT: 'text', 'y': 'label'}), preserve_index=False)

train_ds, val_ds, test_ds = (to_dataset(p).map(tokenize, batched=True) for p in (train_df, val_df, test_df))
too_long = sum(len(ids) > 512 for ids in tokenizer(list(df[TEXT]))['input_ids'])
print(train_ds)
print(f'Passages longer than 512 tokens (truncated): {too_long} of {len(df)}')

**Steps 6 and 7: collator and model.** The data collator pads each batch to its longest passage. We then load DistilBERT with a new classification layer of three outputs, one per class.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

set_seed(SEED)  # the new classification layer gets the same starting weights every run
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=3, id2label=id2label, label2id=label2id, attn_implementation='eager')

**Steps 8 to 10: metric and training arguments.** The trainer reports the macro F1 on the validation years after each epoch and keeps the epoch with the highest value. The settings are those of the book except the learning rate of 5e-5, which gave a clearly higher validation score on these passages than the 1e-5 of the original case, a choice we made on the validation years only.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'macro_f1': f1_score(labels, preds, average='macro'), 'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='ModelOutput',          # where checkpoints are written
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=5e-4,
    eval_strategy='epoch',             # evaluate on the validation years after every epoch
    save_strategy='epoch',
    load_best_model_at_end=True,       # keep the best epoch ...
    metric_for_best_model='macro_f1',  # ... by validation macro F1
    greater_is_better=True,
    logging_strategy='epoch',
    seed=SEED,
    full_determinism=True,             # repeatable on the same GPU type
    report_to='none',
    save_total_limit=2,
)

**Steps 11 to 13: train and save.** Training takes a few minutes on a T4. In the table, the training loss falls steadily, while the validation macro F1 rises for a few epochs and then levels off or wobbles, the usual sign that the model starts to memorize the training years.

In [ ]:
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=data_collator, compute_metrics=compute_metrics)
trainer.train()

trainer.save_model('FEDStanceModel')
tokenizer.save_pretrained('FEDStanceModel')
print('Best validation macro F1:', round(trainer.state.best_metric, 3))

Now the test years, which the model has not seen, neither in training nor in choosing the epoch.

In [ ]:
test_pred = np.argmax(trainer.predict(test_ds).predictions, axis=1)
df.loc[test_df.index, 'pred_distilbert'] = test_pred
evaluate_model('DistilBERT', test_df['y'], test_pred)

## 7. A 2026 encoder: ModernBERT (beyond the book)

DistilBERT dates from 2019. ModernBERT (Warner et al., 2024) is an encoder of the same family trained on far more text, including code and scientific articles, with a native context of 8,192 tokens and faster attention. Swapping it in takes one line: the checkpoint. We keep everything else as above, and only use mixed precision (`fp16`) and gradient accumulation so that it fits and runs fast on a T4.

The run takes several minutes on a T4. Set `RUN_MODERNBERT = False` to skip it.

In [ ]:
RUN_MODERNBERT = True

if RUN_MODERNBERT:
    MB = 'answerdotai/ModernBERT-base'
    mb_tokenizer = AutoTokenizer.from_pretrained(MB)
    mb_tok = lambda batch: mb_tokenizer(batch['text'], truncation=True, max_length=512)
    mb_train, mb_val, mb_test = (to_dataset(p).map(mb_tok, batched=True) for p in (train_df, val_df, test_df))

    set_seed(SEED)
    mb_model = AutoModelForSequenceClassification.from_pretrained(MB, num_labels=3, id2label=id2label, label2id=label2id)
    mb_args = TrainingArguments(
        output_dir='ModernBERTOutput', learning_rate=5e-5,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,   # 32 passages per update, as above
        per_device_eval_batch_size=32, num_train_epochs=15, weight_decay=5e-4,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='macro_f1', greater_is_better=True, logging_strategy='epoch',
        seed=SEED, fp16=torch.cuda.is_available(), report_to='none', save_total_limit=2)
    mb_trainer = Trainer(model=mb_model, args=mb_args, train_dataset=mb_train, eval_dataset=mb_val,
                         data_collator=DataCollatorWithPadding(tokenizer=mb_tokenizer), compute_metrics=compute_metrics)
    mb_trainer.train()
    print('Best validation macro F1:', round(mb_trainer.state.best_metric, 3))
    mb_pred = np.argmax(mb_trainer.predict(mb_test).predictions, axis=1)
    df.loc[test_df.index, 'pred_modernbert'] = mb_pred
    evaluate_model('ModernBERT', test_df['y'], mb_pred)

## 8. Reading the results

The table collects the test scores of every model in this run.

In [ ]:
pd.DataFrame(results).T.round(3).sort_values('macro F1')

Three lessons come out of this table.

**The text has a signal on later years.** Every text model is far above the majority class, on years it never saw. The original case, which predicted an economic index from the same speeches, had no such signal once the split was by time. It makes sense when you think about it: a target that comes from the text, no matter how much [Fedspeak](https://en.wikipedia.org/wiki/Fedspeak) they try to use, such as stance, is much easier to predict for a text model than a target that lives in the economy and depends on how right or wrong the Fed is and how powerful the model is to capture other trends.

**Persistence is a strong baseline.** The label of the previous speech scores well, because stance moves with the policy cycle. A model that does not clearly beat persistence adds little for a bank that already follows the Fed.

**A single run is noisy, and the gains are modest.** On this test set, a difference of a few points of macro F1 is within the noise, because the test years hold about 280 passages, and two runs of the same model with different seeds can differ by three or four points. To give a fairer picture, we (outside this lab, it is a good exercise for you though!) trained each model five times with different seeds, on a larger GPU, with the same data and split. The mean test macro F1 was 0.21 for the majority class, 0.51 for persistence, 0.57 for TF-IDF, 0.57 for DistilBERT and 0.59 for ModernBERT; a majority vote of the five ModernBERT runs reached 0.62. In other words, TF-IDF matches DistilBERT, and ModernBERT is the best model by a small margin that more data or longer passages may widen. Your numbers above will differ somewhat from these, because GPUs and seeds differ.

For a bank, the practical reading is that a modern encoder is worth its cost only where a few points of accuracy on the stance matter, for example in an automated signal for a trading or funding desk. For monitoring and reporting, TF-IDF, or simply the previous label, already gives most of the value. ModernBERT is widely used in low-stakes or low-resource applications though, for example as a compact student that learns financial sentiment from a large language model ([Huang and Simpson, 2026](https://arxiv.org/abs/2606.18875)), as a fast and well-calibrated screen for scam phone calls ([Ren et al., 2026](https://arxiv.org/abs/2609.23959)), or, through encoders of its family, as a cheap safety filter for chatbots such as those of banks, with about 190 times the throughput of a 12-billion-parameter guard model ([Jeon et al., 2026](https://arxiv.org/abs/2606.25782)). Whether you want to make the jump to foundation models, as we do in Chapter 6, is a business decision that depends on the cost of the model, the cost of errors, and the cost of human reading.

## 9. Using the model on new speeches (Algorithm 4.4)

Algorithm 4.4 loads the saved model and tokenizer from disk, prepares a new text in the same way as the training data, and predicts its class. We apply it to the last speeches in the data, from early December 2024, a few weeks before the last FOMC meeting of the year, and to two sentences that you can edit.

In [ ]:
# Load the saved model and tokenizer from disk, as a deployed system would.
stance_model = AutoModelForSequenceClassification.from_pretrained('FEDStanceModel').to(device)
stance_tokenizer = AutoTokenizer.from_pretrained('FEDStanceModel')
stance_model.eval()  # switch off dropout for prediction

def predict_stance(texts):
    """Return the predicted label and the class probabilities for a list of texts."""
    if USE_CLEAN_TEXT:
        texts = [clean_text(t) for t in texts]
    inputs = stance_tokenizer(texts, truncation=True, max_length=512, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = nn.functional.softmax(stance_model(**inputs).logits, dim=1).cpu().numpy()
    return [id2label[i] for i in probs.argmax(1)], probs

In [ ]:
latest = df[df['date'] >= '2024-12-01']
labels, probs = predict_stance(list(latest['passage']))
out = latest[['id', 'date', 'label']].copy()
out['predicted'] = labels
for i, c in enumerate(CLASSES):
    out[f'P({c})'] = probs[:, i].round(2)
out

In [ ]:
# Edit these sentences and run the cell again.
my_sentences = [
    'Inflation remains well above our 2 percent goal, and further increases in the policy rate may be appropriate.',
    'With unemployment rising and inflation close to target, we stand ready to lower the policy rate.',
]
labels, probs = predict_stance(my_sentences)
for s, l, p in zip(my_sentences, labels, probs):
    print(f'{l:8s} {np.round(p, 2)}  {s}')

Compare the predictions with the labels before you trust them. In our runs, the model missed some of the latest passages, and it could get the edited sentences wrong, for example by calling a sentence about rate increases dovish. The counts below show one reason. In the training years, *well above* appears mostly in dovish passages, where it describes unemployment well above its normal level, whereas in the test years it appears mostly in hawkish passages, where it describes inflation well above the Fed's goal. A model learns the words of its training years, and the meaning of words in Fed speeches shifts with the cycle.

In [ ]:
for phrase in ['well above', 'rate increases']:
    table = pd.DataFrame({name: part.loc[part['passage'].str.contains(phrase, case=False), 'label'].value_counts()
                          for name, part in [('train 1997-2016', train_df), ('test 2020-2024', test_df)]}).fillna(0).astype(int)
    print(f'Passages that contain "{phrase}":')
    print(table.reindex([c for c in CLASSES if c in table.index]))
    print()

The model reads a passage the way it was trained: one passage, one label, as of the day of the speech. It does not know what the Fed did afterwards, and its confidence is not a probability of a rate move. In practice, a bank would retrain it as new speeches are labelled, use it to read many speeches quickly and consistently, and check the passages it flags.

## 10. A monthly stance index (beyond the book)

Summing the labels over time gives a simple index: the share of hawkish passages minus the share of dovish ones, by quarter. We plot it against the effective federal funds rate, which we read from FRED, the database of the Federal Reserve Bank of St. Louis. The comparison is descriptive, showing whether the stance of the speeches moves with policy, not whether it predicts it.

In [ ]:
quarterly = (df.set_index('date')['label']
             .groupby(pd.Grouper(freq='QE'))
             .agg(lambda s: (s == 'hawkish').mean() - (s == 'dovish').mean() if len(s) else np.nan))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(quarterly.index, quarterly.values, width=80, color=np.where(quarterly.values > 0, 'tab:red', 'tab:blue'))
ax.set_ylabel('Hawkish share minus dovish share')
ax.axhline(0, color='black', linewidth=0.5)
try:
    ff = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=FEDFUNDS')
    ff.columns = ['date', 'fedfunds']
    ff['date'] = pd.to_datetime(ff['date'])
    ff = ff[(ff['date'] >= '1997-01-01') & (ff['date'] <= '2024-12-31')]
    ax2 = ax.twinx()
    ax2.plot(ff['date'], ff['fedfunds'], color='black', linewidth=1.5, label='Federal funds rate')
    ax2.set_ylabel('Federal funds rate (%)')
    ax2.legend(loc='upper right')
except Exception as e:
    print('FRED could not be reached, so the plot shows the index only:', e)
ax.set_title('Stance of Fed speeches (labels) and the federal funds rate')
plt.tight_layout()
plt.show()

The index turns hawkish ahead of or during each cycle of rate increases and dovish around the cuts of 2001, 2008 and 2020, which suggests that the labels follow policy as intended. It also stays dovish through long periods with the rate at zero, when the Fed eased through asset purchases and guidance instead of the rate.

## 11. Exercises

1. **Preprocessing.** Set `USE_CLEAN_TEXT = True` in Section 3 and run the notebook again. Does Algorithm 4.2 help TF-IDF, DistilBERT, or neither? Then remove the `KEEP` set, so that *not* and *further* are dropped as stopwords, and compare again.
2. **A random split.** Replace the split of Section 4 with a random split (for example, `train_test_split` with 70%, 15% and 15%) and train DistilBERT again. Is the test score higher or lower than with the split by time? Why would a bank trust one more than the other?
3. **Learning rate.** Train DistilBERT with the learning rate of the original case, 1e-5, and compare the validation curve with the one above.
4. **Four classes.** Use `label_4` instead of `label`, so that passages without policy content form their own class, or drop them. How does the confusion matrix change?
5. **Seeds.** Train DistilBERT with three different seeds. How much does the test macro F1 move? Is the difference between DistilBERT and TF-IDF in Section 8 larger than that?
6. **Longer context.** ModernBERT reads up to 8,192 tokens. The 400-word passages do not use that length. Think about how you would build longer inputs from the speeches, and what would change in the labelling.

## What is new in this edition (October 2026)

This edition brings the case study of Chapter 4 up to date. The four algorithms of the chapter are unchanged, whereas the task, the evaluation and the models around them are new. The results are not comparable with those reported in the book. The original case is kept, corrected, in its own notebook (see the link at the top).

1. **A target that lives in the text.** The original case predicted the sign of next month's Chicago Fed National Activity Index from each speech. When the speeches were split by time, that model did no better than chance, and the index of two months earlier predicted the target better than the text, as the usability test at the end of the original case shows. The original case remains a good exercise in how to train a transformer. This edition classifies the monetary policy stance of each speech (hawkish, dovish or neutral), a task that the text can support.
2. **Labels made by a language model and validated against human labels.** The labels were produced with an annotation guide adapted from Shah et al. (2023), by two language models with a third one settling disagreements, and checked against a blind human sample and against the human labels of Shah et al. (Section 2). Their data were used only for that check and are not part of this lab.
3. **Passages instead of whole speeches.** Each speech is reduced to its densest passage on monetary policy, at most 400 words, so that it fits DistilBERT's 512 tokens. Speeches by officials of other central banks were removed.
4. **A split by time.** The models learn from 1997 to 2016, are tuned on 2017 to 2019, and are tested on 2020 to 2024, instead of a random split.
5. **Baselines and macro F1.** The majority class, persistence and TF-IDF with logistic regression are reported next to the transformers, all with macro F1 and confusion matrices, instead of accuracy and AUC.
6. **Preprocessing that keeps negations.** The models read the raw passage by default, and the implementation of Algorithm 4.2 keeps words such as *not* and *further*, which often decide the stance.
7. **A higher learning rate.** 5e-5 instead of 1e-5, chosen on the validation years.
8. **A 2026 encoder.** A new section fine-tunes ModernBERT-base and compares it with DistilBERT.
9. **New inference and an index.** The inference step reads the speeches of December 2024 and sentences you write, and a stance index is set against the federal funds rate.

## References

- Huang, W.-F., and Simpson, E. (2026). Efficient Financial Language Understanding via Distillation with Synthetic Data. [arXiv:2606.18875](https://arxiv.org/abs/2606.18875).
- Jeon, H., Medler, S., Voyles, J., and Wood, M. (2026). Do Encoders Suffice? A Systematic Comparison of Encoder and Decoder Safety Judges for LLM Adversarial Evaluation. [arXiv:2606.25782](https://arxiv.org/abs/2606.25782).
- Ren, S., Zewde, K., Shen, X., et al. (2026). Open-Jev Judgments on CallScreenBench: Calibrated One-Pass Scam Screening with a Small Language Model. [arXiv:2609.23959](https://arxiv.org/abs/2609.23959).
- Sanh, V., Debut, L., Chaumond, J., and Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. [arXiv:1910.01108](https://arxiv.org/abs/1910.01108).
- Shah, A., Paturi, S., and Chava, S. (2023). Trillion Dollar Words: A New Financial Dataset, Task & Market Analysis. *Proceedings of the 61st Annual Meeting of the Association for Computational Linguistics*, 6664-6679. [ACL Anthology](https://aclanthology.org/2023.acl-long.368/).
- Siino, M., Tinnirello, I., and La Cascia, M. (2024). Is text preprocessing still worth the time? A comparative survey on the influence of popular preprocessing methods on Transformers and traditional classifiers. *Information Systems*, 121, 102342. [doi:10.1016/j.is.2023.102342](https://doi.org/10.1016/j.is.2023.102342).
- Warner, B., Chaffin, A., Clavié, B., et al. (2024). Smarter, Better, Faster, Longer: A Modern Bidirectional Encoder for Fast, Memory Efficient, and Long Context Finetuning and Inference. [arXiv:2412.13663](https://arxiv.org/abs/2412.13663).
- Xue, C., Guesmi, R., Feng, S., et al. (2026). Temporal Leakage in Financial News NLP: A Multi-Architecture Audit with a Regime-Specific M&A Signal. [arXiv:2608.17223](https://arxiv.org/abs/2608.17223).